# [Optional/exploratory] Build 1000G reference panel + PCA

**Not part of the main ancestry-filtering pipeline** -- `01_ancestry_pca_filter.ipynb` classifies directly on AoU's own premade PCs (`ancestry_preds.tsv`/`training_pca.tsv`), no 1000G build needed. This notebook exists to build a genuine from-scratch 1000G PCA, which `explore_hm3_ancestry_panel.ipynb` and a 1kG-projection cross-check can use to sanity-check the premade-PC classification against an independent method. CDR/`SAMPLE_SET`-independent -- 1000G doesn't change with either, so this runs once, ever.

Prebuilt 1000G plink files ([plink2 resources](https://www.cog-genomics.org/plink/2.0/resources): 3,202 samples, GRCh38, pgen/pvar/psam) -- no VCF download/merge needed.
Restricted to HapMap3 SNPs (no-MHC list) and the classic 2,504 unrelated samples.

## Setup

plink2: manual install, same pattern as everywhere else in this repo.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version
nproc
free -h

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/1000g_reference"   # no CDR_VERSION -- 1000G itself doesn't depend on it
os.makedirs(BUCKET_DIR, exist_ok=True)

BFILE = f"{BUCKET_DIR}/1kg_all_chrs"                 # HM3-restricted, re-ID'd reference
HM3_SNPLIST = f"{BUCKET_DIR}/hm3.snplist"
PANEL_PATH = f"{BUCKET_DIR}/integrated_call_samples_v3.20130502.ALL.panel"

OUT_PREFIX = f"{BUCKET_DIR}/1kg_all_qc"              # post additional MAF/HWE/geno QC
PRUNE_PREFIX = f"{BUCKET_DIR}/1kg_all_pruned"
PCA_PREFIX = f"{BUCKET_DIR}/1kg_all_pca"

print(BUCKET_DIR)

## Build reference files

Restricted to HapMap3 SNPs (no-MHC list, hosted by the Broad's alkesgroup bucket)
and the classic 2,504 unrelated samples (`PANEL_PATH`, also used later for `pop`/
`super_pop` labels). Downloads/decompresses the ~60 GiB prebuilt pgen directly to
local disk, never touching the gcsfuse-mounted bucket for that file at any point
-- a gcsfuse read of a file this size was found (in this repo's earlier history)
to silently corrupt mid-read regardless of access pattern; only the small final
HM3-restricted result (~340 MiB) gets copied to the bucket.

In [ ]:
%%bash -s "$BUCKET_DIR" "$BFILE" "$HM3_SNPLIST" "$PANEL_PATH"
set -e
BUCKET_DIR=$1
BFILE_PATH=$2
HM3_SNPLIST_PATH=$3
PANEL_PATH=$4

cd "$BUCKET_DIR"

BFILE_NAME=$(basename "$BFILE_PATH")
HM3_NAME=$(basename "$HM3_SNPLIST_PATH")
PANEL_NAME=$(basename "$PANEL_PATH")

command -v zstd >/dev/null 2>&1 || {
  echo "zstd not found -- install it first (e.g. 'sudo apt-get install -y zstd')" >&2
  exit 1
}

fetch() {
  local dest=$1 url=$2
  [ -s "$dest" ] && return 0
  curl -fL --retry 5 --retry-delay 5 --retry-all-errors -o "${dest}.part" "$url"
  mv "${dest}.part" "$dest"
}

# large .zst files from this host are prone to silent truncation mid-transfer --
# check downloaded byte count against the server's real Content-Length, then
# zstd -t as a secondary check (multi-frame files can pass -t even when cut off
# exactly at a frame boundary)
fetch_zst() {
  local dest=$1 url=$2
  [ -s "$dest" ] && return 0
  local want
  want=$(curl -sIL "$url" | awk 'BEGIN{IGNORECASE=1} /^content-length:/{v=$2} END{gsub("","",v); print v}')
  for attempt in 1 2 3; do
    curl -fL --retry 5 --retry-delay 5 --retry-all-errors -o "${dest}.part" "$url"
    local got
    got=$(wc -c < "${dest}.part")
    if [ -n "$want" ] && [ "$got" != "$want" ]; then
      echo "Size mismatch for $dest (got $got, expected $want) on attempt $attempt, retrying" >&2
      rm -f "${dest}.part"
      continue
    fi
    if zstd -t "${dest}.part" 2>/dev/null; then
      mv "${dest}.part" "$dest"
      return 0
    fi
    echo "zstd integrity check failed on attempt $attempt for $dest, retrying" >&2
    rm -f "${dest}.part"
  done
  echo "Failed to download a valid $dest after 3 attempts" >&2
  exit 1
}

fetch "$HM3_NAME" "https://storage.googleapis.com/broad-alkesgroup-public/Variant_effects/1000G_EUR_Phase3_hg38/w_hm3.noMHC.snplist"
fetch "$PANEL_NAME" "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/integrated_call_samples_v3.20130502.ALL.panel"
awk 'NR>1 {print $1}' "$PANEL_NAME" > classic_2504.keep

if [ ! -f "${BFILE_NAME}.bed" ]; then
  LOCAL_WORK_DIR="$HOME/scratch_1kg_hm3"
  mkdir -p "$LOCAL_WORK_DIR"
  cp "$HM3_NAME" classic_2504.keep "$LOCAL_WORK_DIR/"
  cd "$LOCAL_WORK_DIR"

  # Dropbox "scl/fi" share links need both the file id AND rlkey query param.
  # pvar is the "noannot" (rsID-only) build -- much smaller, same variant count
  # as the pgen; the annotated version's link was found to serve a truncated file.
  fetch_zst all_hg38.pgen.zst "https://www.dropbox.com/s/j72j6uciq5zuzii/all_hg38.pgen.zst?dl=1"
  zstd -d --rm all_hg38.pgen.zst
  fetch_zst all_hg38.pvar.zst "https://www.dropbox.com/scl/fi/id642dpdd858uy41og8qi/all_hg38_rs_noannot.pvar.zst?rlkey=sskyiyam1bsqweujjmxqv1h55&dl=1"
  fetch all_hg38.psam "https://www.dropbox.com/scl/fi/u5udzzaibgyvxzfnjcvjc/hg38_corrected.psam?rlkey=oecjnk4vmbhc8b1p202l0ih4x&dl=1"

  plink2     --pgen all_hg38.pgen     --pvar all_hg38.pvar.zst     --psam all_hg38.psam     --keep classic_2504.keep     --max-alleles 2     --rm-dup exclude-all     --extract "$HM3_NAME"     --make-bed     --out 1kg_hm3_raw

  # relabel IDs to chr:pos:ref:alt for matching against the shared ACAF panel later
  plink2     --bfile 1kg_hm3_raw     --set-all-var-ids '@:#:$r:$a'     --new-id-max-allele-len 1000     --make-bed     --out "$BFILE_NAME"

  cd "$BUCKET_DIR"
  cp "$LOCAL_WORK_DIR/${BFILE_NAME}".{bed,bim,fam,log} .
  rm -rf "$LOCAL_WORK_DIR"
fi

wc -l "${BFILE_NAME}.bim" "$HM3_NAME" "${BFILE_NAME}.fam" classic_2504.keep

## Variant QC + LD pruning

No `--keep` -- all 2,504 samples, all populations, already HM3-restricted.
`--hwe 1e-5 0.001 keep-fewhet`: `k=0.001` scales the threshold to this panel's
size (plink2's documented recommendation), `keep-fewhet` only excludes *excess*-
heterozygosity variants so population-structure-driven (Wahlund effect) reduced
heterozygosity in this deliberately multi-population panel isn't mistaken for
genotyping error. `--geno 0.01`/`50 10 0.1` LD window matches this pipeline's
own ACAF QC parameters (`02_genome_wide_qc_thinning_batch_submit.ipynb`).

In [ ]:
%%bash -s "$BFILE" "$OUT_PREFIX" "$PRUNE_PREFIX"
set -e
BFILE=$1
OUT_PREFIX=$2
PRUNE_PREFIX=$3

plink2   --bfile "$BFILE"   --maf 0.01   --hwe 1e-5 0.001 keep-fewhet   --geno 0.01   --max-alleles 2   --rm-dup exclude-all   --nonfounders   --make-bed   --out "$OUT_PREFIX"

plink2   --bfile "$OUT_PREFIX"   --nonfounders   --indep-pairwise 50 10 0.1   --out "$PRUNE_PREFIX"

# Full-panel (not pruned-only) ID+REF+ALT table -- explore_hm3_ancestry_panel.ipynb
# harmonizes ACAF against this directly (same comm -12 pattern a 1kG-projection
# cross-check would use), so it needs REF/ALT for every HM3 QC'd variant, not just the
# pruned subset 1kg_all_pca.acount covers.
plink2   --bfile "$OUT_PREFIX"   --nonfounders   --freq counts   --out "$OUT_PREFIX"

wc -l "${PRUNE_PREFIX}.prune.in" "${OUT_PREFIX}.acount"

## PCA

20 PCs, `allele-wts` -- loadings reused by a 1kG-projection cross-check to project AoU onto this space.

In [ ]:
%%bash -s "$OUT_PREFIX" "$PRUNE_PREFIX" "$PCA_PREFIX"
set -e
OUT_PREFIX=$1
PRUNE_PREFIX=$2
PCA_PREFIX=$3

plink2   --bfile "$OUT_PREFIX"   --extract "${PRUNE_PREFIX}.prune.in"   --nonfounders   --freq counts   --pca allele-wts 20   --out "$PCA_PREFIX"

ls -lh "${PCA_PREFIX}".*

## Per-superpopulation PCA (for round 2's tight fit)

Same fit, restricted to one superpopulation's own individuals -- PC1/PC2 of the
whole-cohort fit above mostly capture continental separation, so within-EUR (or
within-AFR) structure barely shows up there. A 1kG-projection cross-check's
round 2 (tight fit within a matched superpopulation) reads these instead of
`1kg_all_pca`. LD structure differs by population (AFR blocks are much shorter
than EUR's), so each superpop gets its own `--indep-pairwise` pass on its own
samples, not round 1's whole-cohort prune list.

In [ ]:
SUPERPOPS = ["EUR", "AFR"]   # only the families 01_ancestry_pca_filter.ipynb's SAMPLE_SETS actually use
SUPERPOPS_STR = " ".join(SUPERPOPS)

superpop_pca_prefixes = {sp: f"{BUCKET_DIR}/1kg_{sp.lower()}_pca" for sp in SUPERPOPS}
print(superpop_pca_prefixes)

In [ ]:
%%bash -s "$PANEL_PATH" "$OUT_PREFIX" "$BUCKET_DIR" "$SUPERPOPS_STR"
set -e
PANEL_PATH=$1
OUT_PREFIX=$2
BUCKET_DIR=$3
SUPERPOPS=($4)

HEADER=$(head -1 "$PANEL_PATH")
SAMPLE_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'sample' | cut -d: -f1)
SUPERPOP_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'super_pop' | cut -d: -f1)

for SUPERPOP in "${SUPERPOPS[@]}"; do
  SUPERPOP_LOWER=$(echo "$SUPERPOP" | tr '[:upper:]' '[:lower:]')
  KEEP_PATH="${BUCKET_DIR}/1kg_${SUPERPOP_LOWER}.keep"
  PRUNE_PREFIX="${BUCKET_DIR}/1kg_${SUPERPOP_LOWER}_pruned"
  PCA_PREFIX="${BUCKET_DIR}/1kg_${SUPERPOP_LOWER}_pca"

  awk -F'\t' -v sc="$SAMPLE_COL" -v pc="$SUPERPOP_COL" -v pop="$SUPERPOP" \
    'NR>1 && $pc==pop {print $sc}' "$PANEL_PATH" > "$KEEP_PATH"
  echo "${SUPERPOP}: $(wc -l < "$KEEP_PATH") samples"

  plink2 \
    --bfile "$OUT_PREFIX" \
    --keep "$KEEP_PATH" \
    --nonfounders \
    --indep-pairwise 50 10 0.1 \
    --out "$PRUNE_PREFIX"

  plink2 \
    --bfile "$OUT_PREFIX" \
    --keep "$KEEP_PATH" \
    --extract "${PRUNE_PREFIX}.prune.in" \
    --nonfounders \
    --freq counts \
    --pca allele-wts 20 \
    --out "$PCA_PREFIX"
done

ls -lh "${BUCKET_DIR}"/1kg_*_pca.* "${BUCKET_DIR}"/1kg_*_pruned.*